# Comparativa 6 tecnicas de tokenizacion - Weather (modelo reducido)
Misma arquitectura para todas: d_model=16, n_heads=2, e_layers=1, d_ff=32.
Cada celda = 1 tecnica. Si ya existe checkpoint, skip.

In [ ]:
import os, subprocess, numpy as np

os.chdir('/home/jaime/TFG/RITMO')

TECHNIQUES = ['discretization', 'text_based', 'patching', 'decomposition', 'foundation', 'hmm', 'hmm_soft', 'hmm_soft_residual']

BASE_CMD = [
    'python', '-u', 'run.py',
    '--task_name', 'plan_a',
    '--is_training', '1',
    '--root_path', './dataset/weather/',
    '--data_path', 'weather.csv',
    '--model_id', 'Weather_96_96',
    '--model', 'TransformerCommon',
    '--data', 'Weather',
    '--features', 'S',
    '--seq_len', '96',
    '--pred_len', '96',
    '--enc_in', '1',
    '--dec_in', '1',
    '--c_out', '1',
    '--d_model', '16',
    '--n_heads', '2',
    '--e_layers', '1',
    '--d_ff', '32',
    '--dropout', '0.1',
    '--batch_size', '32',
    '--learning_rate', '0.0001',
    '--train_epochs', '20',
    '--patience', '5',
    '--use_gpu', '0',
    '--itr', '1',
]

def setting_for(technique, K=5):
    des = f'Plan_A_{technique}' + (f'_K{K}' if technique.startswith('hmm') else '')
    return f'plan_a_Weather_96_96_TransformerCommon_Weather_ftS_sl96_ll48_pl96_dm16_nh2_el1_dl1_df32_expand2_dc4_fc1_ebtimeF_dtTrue_{des}_0'

def run_technique(technique, K=5):
    ckpt = f'./checkpoints/{setting_for(technique, K)}/checkpoint.pth'
    if os.path.exists(ckpt):
        print(f'{technique}: SKIP - checkpoint ya existe')
        return
    des = f'Plan_A_{technique}' + (f'_K{K}' if technique.startswith('hmm') else '')
    cmd = BASE_CMD + ['--technique', technique, '--hmm_k', str(K), '--des', des]
    print(f'{technique}: ejecutando...')
    r = subprocess.run(cmd, capture_output=False)
    print(f'{technique}: {"OK" if r.returncode == 0 else "ERROR"}')

print('Funcion run_technique(technique, K=5) lista.')

In [ ]:
run_technique('discretization')

In [ ]:
run_technique('text_based')

In [ ]:
run_technique('patching')

In [ ]:
run_technique('decomposition')

In [ ]:
run_technique('foundation')

In [ ]:
run_technique('hmm', K=5)

In [ ]:
run_technique('hmm_soft', K=5)

In [ ]:
run_technique('hmm_soft_residual', K=5)

In [ ]:
# Resumen de resultados
print(f'{"Tecnica":>20s} | {"MSE":>10} | {"MAE":>10}')
print('-' * 48)
for tech in TECHNIQUES:
    K = 5
    setting = setting_for(tech, K)
    metrics_path = f'./results/{setting}/metrics.npy'
    if os.path.exists(metrics_path):
        mae, mse, rmse, mape, mspe = np.load(metrics_path)
        print(f'{tech:>20s} | {mse:10.6f} | {mae:10.6f}')
    else:
        print(f'{tech:>20s} | {"pendiente":>10} | {"pendiente":>10}')